In [ ]:
# Importando bibliotecas para a criação do relatório da AUDITORIA INTERNA
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# =========================================================
# 1) LOAD + FILTRO
# =========================================================

df_ronda = pd.read_parquet(r"..\Analisando Irregularidades dos Registros\df_ronda.parquet")

# Garantir tipos ANTES do filtro
df_ronda["data"] = pd.to_datetime(df_ronda["data"], errors="coerce").dt.normalize()
df_ronda["planta"] = df_ronda["planta"].astype(str)

# Se ainda não existir, cria "area" pra não quebrar
if "area" not in df_ronda.columns:
    df_ronda["area"] = "Sem Área"
df_ronda["area"] = df_ronda["area"].fillna("Sem Área")

# Semanas analisadas
semanas = [
    ("2025-04-28", "2025-05-04"),  # 01/05 (qui) — Dia do Trabalho
    ("2025-06-16", "2025-06-22"),  # 19/06 (qui) — Corpus Christi
    ("2025-11-17", "2025-11-23"),  # 20/11 (qui) — Consciência Negra
    ("2025-12-22", "2025-12-28"),  # 25/12 (qui) — Natal
    ("2026-02-16", "2026-02-22"),  # 17/02 (ter) — Carnaval
    ("2026-04-20", "2026-04-26"),  # 21/04 (ter) — Tiradentes
    ("2026-06-01", "2026-06-07"),  # 04/06 (qui) — Corpus Christi
]

# Filtro por herança + semanas
mask_periodo = pd.Series(False, index=df_ronda.index)
for ini, fim in semanas:
    ini = pd.to_datetime(ini)
    fim = pd.to_datetime(fim)
    mask_periodo = mask_periodo | df_ronda["data"].between(ini, fim)

df_ronda = df_ronda.loc[
    (df_ronda["heranca"] == "AUDITORIA INTERNA") & mask_periodo
].copy()

# Tag de semana (início)
conds = [df_ronda["data"].between(pd.to_datetime(ini), pd.to_datetime(fim)) for ini, fim in semanas]
week_starts = [pd.to_datetime(ini) for ini, _ in semanas]
df_ronda["semana_inicio"] = np.select(conds, week_starts, default=pd.NaT)

# Presencial vs Home (planta 999 = home)
df_ronda["presencial"] = df_ronda["planta"] != "999"

# =========================================================
# 2) RESUMO
# =========================================================
gcols = ["cadastro", "nome", "area", "semana_inicio"]

dias_com_registro = df_ronda.groupby(gcols)["data"].nunique().rename("dias_com_registro")

dias_presenciais = (
    df_ronda[df_ronda["presencial"]]
    .groupby(gcols)["data"].nunique().rename("dias_presenciais")
)

dias_home = (
    df_ronda[~df_ronda["presencial"]]
    .groupby(gcols)["data"].nunique().rename("dias_home")
)

resumo = (
    pd.concat([dias_com_registro, dias_presenciais, dias_home], axis=1)
      .fillna({"dias_com_registro": 0, "dias_presenciais": 0, "dias_home": 0})
      .reset_index()
)

resumo[["dias_com_registro", "dias_presenciais", "dias_home"]] = resumo[
    ["dias_com_registro", "dias_presenciais", "dias_home"]
].astype(int)

resumo["semana_fim"] = resumo["semana_inicio"] + pd.Timedelta(days=6)

# Quem NÃO cumpriu 3 dias presenciais (IGNORANDO quem teve 0 presencial)
nao_cumpriu = (
    resumo[(resumo["dias_presenciais"] < 3) & (resumo["dias_presenciais"] > 0)]
      .sort_values(["semana_inicio", "dias_presenciais", "cadastro"])
      .reset_index(drop=True)
)

# =========================================================
# >>> ALTERAÇÃO PRINCIPAL: NÃO contar/plotar dias_presenciais == 0 nos VISUAIS
# =========================================================

# Flag consistente com a regra do nao_cumpriu (zero NÃO entra como não conformidade)
resumo["nao_cumpriu_3"] = (resumo["dias_presenciais"] < 3) & (resumo["dias_presenciais"] > 0)

# semana stats (gráfico D) — calculado na base sem zeros
semana_stats = (
    resumo_vis.groupby("semana_inicio", as_index=False)
       .agg(
           pessoa_semanas=("cadastro", "size"),
           pessoas_unicas=("cadastro", "nunique"),
           pct_nao_cumpriu_3=("nao_cumpriu_3", "mean"),
           media_dias_presenciais=("dias_presenciais", "mean"),
       )
)

# =========================================================
# 3) DASHBOARD (8 GRÁFICOS)
# =========================================================

# Paleta “Volks-like”
VW_NAVY   = "#001E50"
VW_BLUE   = "#0B5FA5"
VW_BLUE2  = "#7FB0D6"
VW_GREY   = "#C9D1D9"
VW_DARK   = "#1B2A41"
ACCENT    = "#FFB000"
BG_LIGHT  = "#F6F8FB"

plt.rcParams.update({
    "figure.facecolor": BG_LIGHT,
    "axes.facecolor": "white",
    "axes.edgecolor": VW_GREY,
    "axes.labelcolor": VW_DARK,
    "xtick.color": VW_DARK,
    "ytick.color": VW_DARK,
    "text.color": VW_DARK,
    "font.size": 10,
})

VW_CMAP = LinearSegmentedColormap.from_list(
    "vw_blues",
    [BG_LIGHT, "#CFE2F3", VW_BLUE, VW_NAVY]
)

def add_desc(ax, text):
    ax.text(
        0.01, -0.35, text,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=9, color="#334155",
        wrap=True,
        bbox=dict(boxstyle="round,pad=0.45", facecolor=BG_LIGHT, edgecolor=VW_GREY, linewidth=0.9)
    )

def style_card(ax):
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_color(VW_GREY)
        s.set_linewidth(1.2)
    ax.set_facecolor("white")

def make_first_name_labels(series_names: pd.Series) -> pd.Series:
    first = series_names.astype(str).str.strip().str.split().str[0].replace({"": "Semnome"})
    counts = {}
    labels = []
    for fn in first.tolist():
        counts[fn] = counts.get(fn, 0) + 1
        labels.append(fn if counts[fn] == 1 else f"{fn} ({counts[fn]})")
    return pd.Series(labels, index=series_names.index)


# =========================================================
# DESCRIÇÕES (mais interpretáveis / focadas no "o que significa")
# =========================================================
DESC = {
    "A": (
        "Resumo:\n"
        "• Existe uma área que puxa o problema: ela aparece com a maior taxa de semanas <3 dias.\n"
        "• Como os volumes entre áreas são parecidos, a diferença real é o comportamento (taxa), não o tamanho.\n"
        "O que isso sugere:\n"
        "• Prioridade é atuar nessa área líder (é o maior “risco” do painel).\n"
        "Próximo passo:\n"
        "• Quebrar essa área por gestor/time/turno para encontrar o foco exato."
    ),
    "B": (
        "Resumo:\n"
        "• O volume total de não conformidade está concentrado em 1 área (as outras contribuem bem menos).\n"
        "O que isso sugere:\n"
        "• Se você atacar a 1ª área do ranking, você reduz a maior parte do problema mais rápido.\n"
        "Próximo passo:\n"
        "• Tratar como plano 80/20: ação imediata na líder, monitorar as demais."
    ),
    "C": (
        "Resumo:\n"
        "• Uma área mostra instabilidade (varia bastante entre semanas e chega a ter pico muito alto).\n"
        "• Outras áreas têm problema pontual (aparece em uma semana e some nas demais).\n"
        "O que isso sugere:\n"
        "• A principal área tem padrão recorrente; as demais parecem efeito de evento específico.\n"
        "Próximo passo:\n"
        "• Na semana do pico: cruzar com agenda, férias, viagens, reuniões externas e comunicação interna."
    ),
    "D": (
        "Resumo:\n"
        "• Quando a média de dias presenciais cai, o % de não conformidade tende a subir (e vice-versa).\n"
        "• Existe uma semana claramente pior que as outras (pico de não conformidade).\n"
        "O que isso sugere:\n"
        "• O problema é sensível a contexto/rotina da semana (evento, agenda, exceções).\n"
        "Próximo passo:\n"
        "• Usar essa semana como “caso” para investigação (o que mudou nela?)."
    ),
    "E": (
        "Resumo:\n"
        "• A maioria está no alvo (3 dias), e a segunda maior fatia fica muito perto (2 dias).\n"
        "• Casos extremos (1 dia) são poucos.\n"
        "O que isso sugere:\n"
        "• O ganho mais rápido é converter quem está em 2 dias para 3 dias (ajuste fino).\n"
        "Próximo passo:\n"
        "• Direcionar comunicação/combinação de agenda para o grupo de 2 dias."
    ),
    "F": (
        "Resumo:\n"
        "• Aparecem poucos Nomes com recorrência (repetem <3 dias em mais de uma semana).\n"
        "• O restante tende a ser casos pontuais (uma ocorrência).\n"
        "O que isso sugere:\n"
        "• Você tem “perfil recorrente” (merece checagem de exceção) e “perfil pontual” (evento).\n"
        "Próximo passo:\n"
        "• Recorrentes: validar justificativa/arranjo formal; Pontuais: cruzar com a semana do pico."
    ),
    "G": (
        "Resumo:\n"
        "• Mostra quem tem maior presença acumulada no período (referência de comportamento).\n"
        "O que isso sugere:\n"
        "• Serve como benchmark de rotina/agenda que funciona.\n"
        "Próximo passo:\n"
        "• Comparar a rotina dos mais presentes com a área crítica (o que eles fazem diferente?)."
    ),
    "H": (
        "Resumo:\n"
        "• Mostra quem é consistente (cumpre em várias semanas, não só “média alta”).\n"
        "O que isso sugere:\n"
        "• Melhor indicador de regularidade do que o total (G).\n"
        "Próximo passo:\n"
        "• Replicar boas práticas dos consistentes na área crítica (agenda, rituais, acordos de presença)."
    ),
}


# Base estatística pro dashboard (já sem zeros via resumo_vis)
area_agg = (
    resumo_vis.groupby("area", dropna=False, as_index=False)
       .agg(
           pessoa_semanas=("semana_inicio", "size"),
           pessoas_unicas=("cadastro", "nunique"),
           nao_cumpriu_count=("nao_cumpriu_3", "sum"),
           pct_nao_cumpriu=("nao_cumpriu_3", "mean"),
           media_dias_presenciais=("dias_presenciais", "mean"),
       )
)
area_agg["score_risco"] = area_agg["pct_nao_cumpriu"] * np.sqrt(area_agg["pessoa_semanas"])

MIN_PESSOA_SEMANAS = 10
area_base = area_agg[area_agg["pessoa_semanas"] >= MIN_PESSOA_SEMANAS].copy()
area_top = (
    area_base.sort_values(["score_risco","pct_nao_cumpriu","pessoa_semanas"], ascending=[False, False, False])
             .head(10)
             .copy()
)

pessoa_stats = (
    resumo_vis.groupby(["cadastro","nome","area"], as_index=False)
       .agg(
           semanas_total=("semana_inicio","nunique"),
           semanas_nao_cumpriu=("nao_cumpriu_3","sum"),
           dias_home_total=("dias_home","sum"),
           dias_presenciais_total=("dias_presenciais","sum"),
       )
)
pessoa_stats["taxa_nao_cumpriu"] = pessoa_stats["semanas_nao_cumpriu"] / pessoa_stats["semanas_total"] * 100
pessoa_stats["semanas_cumpriu"] = pessoa_stats["semanas_total"] - pessoa_stats["semanas_nao_cumpriu"]
pessoa_stats["taxa_cumpriu"] = pessoa_stats["semanas_cumpriu"] / pessoa_stats["semanas_total"] * 100

# Painel 4x2
fig = plt.figure(figsize=(18, 26))
gs = fig.add_gridspec(4, 2, hspace=0.85, wspace=0.28)

# ---------------------------------------------------------
# (A) Prioridade — Taxa x Volume
# ---------------------------------------------------------
axA = fig.add_subplot(gs[0, 0]); style_card(axA)

dfA = area_base.copy()
dfA["pct"] = dfA["pct_nao_cumpriu"] * 100
x_med = dfA["pessoa_semanas"].median()
y_med = dfA["pct"].median()
topN = dfA.sort_values("score_risco", ascending=False).head(8)

axA.scatter(dfA["pessoa_semanas"], dfA["pct"],
            s=(dfA["pessoas_unicas"] * 10).clip(40, 900),
            alpha=0.65, color=VW_BLUE2, edgecolors=VW_NAVY, linewidths=0.6)
axA.scatter(topN["pessoa_semanas"], topN["pct"],
            s=(topN["pessoas_unicas"] * 12).clip(70, 1100),
            alpha=0.95, color=ACCENT, edgecolors=VW_NAVY, linewidths=0.8)

for _, r in topN.iterrows():
    axA.annotate(str(r["area"]),
                 xy=(r["pessoa_semanas"], r["pct"]),
                 xytext=(18, 14),
                 textcoords="offset points",
                 ha="left", va="bottom", fontsize=8,
                 bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor=VW_GREY, linewidth=0.6, alpha=0.9))

axA.axvline(x_med, linestyle="--", color=VW_GREY, linewidth=1)
axA.axhline(y_med, linestyle="--", color=VW_GREY, linewidth=1)
axA.set_title("A) Prioridade — Taxa × Volume por Área", pad=10, color=VW_NAVY, fontweight="bold")
axA.set_xlabel("Volume (pessoa-semanas)")
axA.set_ylabel("% com < 3 dias presenciais")
axA.grid(True, linestyle="--", alpha=0.18)

axA.legend(handles=[
    Line2D([0],[0], marker='o', color='w', label='Áreas (geral)',
           markerfacecolor=VW_BLUE2, markeredgecolor=VW_NAVY, markersize=9),
    Line2D([0],[0], marker='o', color='w', label='Top 8 (maior risco ajustado por volume)',
           markerfacecolor=ACCENT, markeredgecolor=VW_NAVY, markersize=9),
], loc="upper left", frameon=True, facecolor="white", edgecolor=VW_GREY)

add_desc(axA, DESC["A"])

# ---------------------------------------------------------
# (B) Pareto — SOMENTE BARRAS
# ---------------------------------------------------------
axB = fig.add_subplot(gs[0, 1]); style_card(axB)

dfB = area_base.sort_values("nao_cumpriu_count", ascending=False).head(12).copy()
dfB["taxa"] = (dfB["pct_nao_cumpriu"] * 100).round(1)

axB.bar(dfB["area"].astype(str), dfB["nao_cumpriu_count"], color=VW_NAVY, alpha=0.92)
axB.set_title("B) Pareto — Massa de não conformidade (volume absoluto)", pad=10, color=VW_NAVY, fontweight="bold")
axB.set_xlabel("Área")
axB.set_ylabel("Qtd de pessoa-semanas < 3 dias")
axB.grid(axis="y", linestyle="--", alpha=0.18)
axB.tick_params(axis="x", rotation=35)

max_bar = float(dfB["nao_cumpriu_count"].max())
axB.set_ylim(0, max_bar * 1.35)

for i, r in enumerate(dfB.itertuples(index=False)):
    axB.text(i, r.nao_cumpriu_count + max_bar*0.03, f"{int(r.nao_cumpriu_count)}\n({r.taxa}%)",
             ha="center", va="bottom", fontsize=8, color=VW_DARK)

legendB = [Patch(facecolor=VW_NAVY, edgecolor=VW_NAVY, label="Volume: pessoa-semanas < 3 dias (barra)")]
axB.legend(handles=legendB, loc="upper right", frameon=True, facecolor="white", edgecolor=VW_GREY)

add_desc(axB, DESC["B"])

# ---------------------------------------------------------
# (C) Heatmap — % não cumpriu por Área × semana (Top 10)
# ---------------------------------------------------------
axC = fig.add_subplot(gs[1, 0]); style_card(axC)

aw = (resumo_vis[resumo_vis["area"].astype(str).isin(area_top["area"].astype(str))]
      .groupby(["area","semana_inicio"], as_index=False)["nao_cumpriu_3"]
      .mean()
      .rename(columns={"nao_cumpriu_3": "pct"}))
aw["pct"] *= 100

pivot = aw.pivot_table(index="area", columns="semana_inicio", values="pct", aggfunc="mean")
pivot = pivot.reindex(area_top["area"].astype(str).tolist())
pivot = pivot.reindex(sorted(pivot.columns), axis=1)

mat = pivot.values
im = axC.imshow(mat, aspect="auto", cmap=VW_CMAP, vmin=0, vmax=100)
axC.set_title("C) Estabilidade — % não cumpriu por Área × semana (Top 10)", pad=10, color=VW_NAVY, fontweight="bold")
axC.set_xlabel("semana (início)")
axC.set_ylabel("Área")

xlabels = [pd.to_datetime(c).strftime("%Y-%m-%d") for c in pivot.columns]
axC.set_xticks(np.arange(len(xlabels)))
axC.set_xticklabels(xlabels, rotation=45, ha="right")
axC.set_yticks(np.arange(pivot.shape[0]))
axC.set_yticklabels(pivot.index.astype(str))

for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i, j]
        if np.isfinite(v):
            axC.text(j, i, f"{v:.0f}%", ha="center", va="center",
                     fontsize=8, color=("white" if v >= 55 else VW_DARK))

cbar = fig.colorbar(im, ax=axC, fraction=0.046, pad=0.04)
cbar.set_label("% não cumprimento (<3 dias)")

add_desc(axC, DESC["C"])

# ---------------------------------------------------------
# (D) Tendência semanal — % não cumpriu vs média presencial
# ---------------------------------------------------------
axD = fig.add_subplot(gs[1, 1]); style_card(axD)

dfD = semana_stats.sort_values("semana_inicio").copy()
dfD["semana"] = dfD["semana_inicio"].dt.strftime("%Y-%m-%d")

pct = (dfD["pct_nao_cumpriu_3"] * 100).to_numpy()
media = dfD["media_dias_presenciais"].to_numpy()

l1, = axD.plot(dfD["semana"], pct, marker="o", color=VW_NAVY, linewidth=2.4, label="% não cumpriu")
axD.set_title("D) Evolução semanal — % não cumpriu vs média presencial", pad=10, color=VW_NAVY, fontweight="bold")
axD.set_xlabel("semana (início)")
axD.set_ylabel("% não cumpriu")
axD.grid(True, linestyle="--", alpha=0.18)
axD.tick_params(axis="x", rotation=45)

axD2 = axD.twinx()
l2, = axD2.plot(dfD["semana"], media, marker="o", color=VW_BLUE, linewidth=2.2, label="Média dias presenciais")
axD2.set_ylabel("Média dias presenciais")

axD.legend(handles=[l1, l2], loc="upper center", frameon=True, facecolor="white", edgecolor=VW_GREY)

for x, y in zip(dfD["semana"], pct):
    axD.text(x, y + 0.6, f"{y:.1f}%", ha="center", fontsize=8)

add_desc(axD, DESC["D"])

# ---------------------------------------------------------
# (E) Severidade — distribuição de dias presenciais (sem 0)
# ---------------------------------------------------------
axE = fig.add_subplot(gs[2, 0]); style_card(axE)

dist = (resumo_vis.groupby("dias_presenciais").size().rename("qtd").reset_index().sort_values("dias_presenciais"))
dist["pct"] = dist["qtd"] / dist["qtd"].sum() * 100

axE.bar(dist["dias_presenciais"].astype(int).astype(str), dist["pct"], color=VW_BLUE, alpha=0.92)
axE.set_title("E) Severidade — Distribuição de dias presenciais", pad=10, color=VW_NAVY, fontweight="bold")
axE.set_xlabel("Dias presenciais na semana")
axE.set_ylabel("% de pessoa-semanas")
axE.grid(axis="y", linestyle="--", alpha=0.18)

ymaxE = float(dist["pct"].max()) * 1.25
axE.set_ylim(0, ymaxE)
for i, r in dist.iterrows():
    axE.text(i, r["pct"] + ymaxE*0.02, f"{r['pct']:.1f}%", ha="center", fontsize=8)

add_desc(axE, DESC["E"])

# ---------------------------------------------------------
# (F) Investigação — recorrência (<3 dias) por pessoa
# ---------------------------------------------------------
axF = fig.add_subplot(gs[2, 1]); style_card(axF)

topF = (pessoa_stats.sort_values(["semanas_nao_cumpriu","taxa_nao_cumpriu","dias_home_total"],
                                ascending=[False, False, False])
                    .head(12)
                    .copy())
topF["primeiro_nome"] = make_first_name_labels(topF["nome"])

valsF = topF["semanas_nao_cumpriu"].to_numpy()
axF.bar(topF["primeiro_nome"], valsF, color=VW_NAVY, alpha=0.92)
axF.set_title("F) Investigação — Recorrência (<3 dias) por pessoa", pad=10, color=VW_NAVY, fontweight="bold")
axF.set_xlabel("Primeiro nome")
axF.set_ylabel("Qtd de semanas < 3 dias")
axF.grid(axis="y", linestyle="--", alpha=0.18)

axF.tick_params(axis="x", rotation=45)
for lab in axF.get_xticklabels():
    lab.set_ha("right")

axF.set_ylim(0, max(valsF.max()*1.35, valsF.max()+1))
for i, r in enumerate(topF.itertuples(index=False)):
    axF.text(i, r.semanas_nao_cumpriu + 0.08, f"{str(r.area)}\n{r.taxa_nao_cumpriu:.0f}%",
             ha="center", va="bottom", fontsize=8)

add_desc(axF, DESC["F"])

# ---------------------------------------------------------
# (G) Quem mais vem — dias presenciais total
# ---------------------------------------------------------
axG = fig.add_subplot(gs[3, 0]); style_card(axG)

topG = (pessoa_stats.sort_values(["dias_presenciais_total","semanas_cumpriu","taxa_cumpriu"],
                                ascending=[False, False, False])
                    .head(12)
                    .copy())
topG["primeiro_nome"] = make_first_name_labels(topG["nome"])

valsG = topG["dias_presenciais_total"].to_numpy()
axG.bar(topG["primeiro_nome"], valsG, color=VW_BLUE, alpha=0.92)
axG.set_title("G) Quem mais vem — Dias presenciais (total)", pad=10, color=VW_NAVY, fontweight="bold")
axG.set_xlabel("Primeiro nome")
axG.set_ylabel("Dias presenciais (soma)")
axG.grid(axis="y", linestyle="--", alpha=0.18)

axG.tick_params(axis="x", rotation=45)
for lab in axG.get_xticklabels():
    lab.set_ha("right")

axG.set_ylim(0, max(valsG.max()*1.25, valsG.max()+1))
for i, r in enumerate(topG.itertuples(index=False)):
    axG.text(i, r.dias_presenciais_total + 0.2, f"{str(r.area)}\n{r.taxa_cumpriu:.0f}%",
             ha="center", va="bottom", fontsize=8)

add_desc(axG, DESC["G"])

# ---------------------------------------------------------
# (H) Quem mais vem — consistência (semanas cumprindo)
# ---------------------------------------------------------
axH = fig.add_subplot(gs[3, 1]); style_card(axH)

baseH = pessoa_stats[pessoa_stats["semanas_total"] >= 2].copy()
topH = (baseH.sort_values(["semanas_cumpriu","taxa_cumpriu","dias_presenciais_total"],
                         ascending=[False, False, False])
             .head(12)
             .copy())
topH["primeiro_nome"] = make_first_name_labels(topH["nome"])

valsH = topH["semanas_cumpriu"].to_numpy()
axH.bar(topH["primeiro_nome"], valsH, color=VW_NAVY, alpha=0.85)
axH.set_title("H) Quem mais vem — Consistência (semanas ≥3 dias)", pad=10, color=VW_NAVY, fontweight="bold")
axH.set_xlabel("Primeiro nome")
axH.set_ylabel("Qtd semanas cumprindo")
axH.grid(axis="y", linestyle="--", alpha=0.18)

axH.tick_params(axis="x", rotation=45)
for lab in axH.get_xticklabels():
    lab.set_ha("right")

axH.set_ylim(0, max(valsH.max()*1.25, valsH.max()+1))
for i, r in enumerate(topH.itertuples(index=False)):
    axH.text(i, r.semanas_cumpriu + 0.08, f"{str(r.area)}\n{r.taxa_cumpriu:.0f}%",
             ha="center", va="bottom", fontsize=8)

add_desc(axH, DESC["H"])

fig.suptitle(
    "Painel de Compliance dos dias Presenciais — Auditoria Interna (8 insights)",
    fontsize=20, fontweight="bold", color=VW_NAVY, y=0.92
)

# =========================================================
# 4) EXPORT PNG + PDF + SAÍDAS
# =========================================================
OUT_PDF = "Relatório da Auditoria Interna.pdf"

# PDF com o painel (mesma figura)
fig.savefig(OUT_PDF, bbox_inches="tight")

plt.show()

print((nao_cumpriu.groupby('nome').size().reset_index(name='qtd_aparicoes')).to_string(index=False))

print(nao_cumpriu.to_string(index=False))